In [ ]:
!pip install pandas sqlalchemy pymysql

In [18]:
import pandas as pd

# Load the dataset from the local CSV file
df = pd.read_csv('shopping_table.csv')

# Display the first 5 rows to confirm it loaded correctly
display(df.head())

,Customer_ID,Age,Gender,Item_Purchased,Category,Purchase_Amount_USD,Location,Size,Color,Season,Review_Rating,Subscription_Status,Payment_Method,Shipping_Type,Discount_Applied,Promo_Code_Used,Previous_Purchases,Preferred_Payment_Method,Frequency_of_Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Credit Card,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Bank Transfer,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Cash,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,PayPal,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Cash,Free Shipping,Yes,Yes,31,PayPal,Annually


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_ID               3900 non-null   int64  
 1   Age                       3900 non-null   int64  
 2   Gender                    3900 non-null   object 
 3   Item_Purchased            3900 non-null   object 
 4   Category                  3900 non-null   object 
 5   Purchase_Amount_USD       3900 non-null   int64  
 6   Location                  3900 non-null   object 
 7   Size                      3900 non-null   object 
 8   Color                     3900 non-null   object 
 9   Season                    3900 non-null   object 
 10  Review_Rating             3900 non-null   float64
 11  Subscription_Status       3900 non-null   object 
 12  Payment_Method            3900 non-null   object 
 13  Shipping_Type             3900 non-null   object 
 14  Discount

In [20]:
df.columns

Index(['Customer_ID', 'Age', 'Gender', 'Item_Purchased', 'Category',
       'Purchase_Amount_USD', 'Location', 'Size', 'Color', 'Season',
       'Review_Rating', 'Subscription_Status', 'Payment_Method',
       'Shipping_Type', 'Discount_Applied', 'Promo_Code_Used',
       'Previous_Purchases', 'Preferred_Payment_Method',
       'Frequency_of_Purchases'],
      dtype='object')

In [21]:
df.isnull().sum()

,0
Customer_ID,0
Age,0
Gender,0
Item_Purchased,0
Category,0
Purchase_Amount_USD,0
Location,0
Size,0
Color,0
Season,0


In [22]:
# 1. See how many items were sold in each category
category_counts = df['Category'].value_counts()
print("Sales Volume by Category:")
print(category_counts)

# 2. See the total revenue for each category
category_revenue = df.groupby('Category')['Purchase_Amount_USD'].sum().sort_values()
print("\nTotal Revenue by Category:")
print(category_revenue)

Sales Volume by Category:
Category
Clothing       1737
Accessories    1240
Footwear        599
Outerwear       324
Name: count, dtype: int64

Total Revenue by Category:
Category
Outerwear       18524
Footwear        36093
Accessories     74200
Clothing       104264
Name: Purchase_Amount_USD, dtype: int64


In [23]:
# 1. Create Age Groups to make analysis easier
bins = [18, 25, 35, 45, 55, 100]
labels = ['18-25', '26-35', '36-45', '46-55', '55+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

# 2. Check total/average spending by Age Group
age_spending = df.groupby('Age_Group')['Purchase_Amount_USD'].mean()
print("Average Spending by Age Group:")
print(age_spending)

# 3. Explore the impact of Season on spending
season_spending = df.groupby('Season')['Purchase_Amount_USD'].mean().sort_values(ascending=False)
print("\nAverage Spending by Season:")
print(season_spending)

# 4. Explore the impact of Location on spending (Top 5 locations)
location_spending = df.groupby('Location')['Purchase_Amount_USD'].mean().sort_values(ascending=False).head()
print("\nTop 5 Locations by Average Spending:")
print(location_spending)

Average Spending by Age Group:
Age_Group
18-25    60.201646
26-35    60.132450
36-45    59.620027
46-55    60.332447
55+      59.074703
Name: Purchase_Amount_USD, dtype: float64

Average Spending by Season:
Season
Fall      61.556923
Winter    60.357364
Spring    58.737738
Summer    58.405236
Name: Purchase_Amount_USD, dtype: float64

Top 5 Locations by Average Spending:
Location
Alaska           67.597222
Pennsylvania     66.567568
Arizona          66.553846
West Virginia    63.876543
Nevada           63.379310
Name: Purchase_Amount_USD, dtype: float64


/tmp/ipykernel_4961/3639956567.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_spending = df.groupby('Age_Group')['Purchase_Amount_USD'].mean()


In [24]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [25]:
# 1. Select the numerical features for clustering
features = ['Age', 'Purchase_Amount_USD', 'Previous_Purchases']
X = df[features].dropna() # Dropping any empty rows just in case

In [26]:
# 2. Scale the data (Crucial for K-Means to work properly)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [27]:
# 3. Apply K-Means Clustering
# The task asks for 3 or more groups. Let's start with 3.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

In [28]:
# 4. Analyze the Clusters to identify dominant factors
# We look at the average values for each feature within each newly created cluster
cluster_analysis = df.groupby('Cluster')[features].mean()
print("Dominant Factors per Cluster (Averages):")
display(cluster_analysis)

Dominant Factors per Cluster (Averages):


,Age,Purchase_Amount_USD,Previous_Purchases
Cluster,,,
0,39.956053,73.030680,11.561360
1,45.792105,35.646053,26.443421
2,46.061329,77.362862,38.103918


In [29]:
# 5. See how many customers fell into each cluster
print("\nNumber of Customers in each Cluster:")
print(df['Cluster'].value_counts())


Number of Customers in each Cluster:
Cluster
1    1520
0    1206
2    1174
Name: count, dtype: int64
